# 01 — Decision Intelligence Overview
## bd_replica_crm · dato → predicción → recomendación → acción → outcome

Este notebook responde cinco preguntas:

1. **¿Qué está ocurriendo comercialmente?**
2. **¿Qué está prediciendo el modelo?**
3. **¿Qué recomendaciones genera?**
4. **¿El equipo registra acciones sobre esas recomendaciones?**
5. **¿Qué outcomes observamos cuando los horizontes maduran?**

El foco no es demostrar causalidad todavía. El foco es demostrar un sistema **trazable, medible y accionable**.


### Modelo conceptual

```text
CRM / ciclo comercial
        ↓
features.lead_evidence
        ↓
decision_intelligence.lead_scores
        ↓
decision_intelligence.recommendations
        ↓
decision_intelligence.actions
        ↓
decision_intelligence.outcomes
        ↓
aprendizaje / experimento / nueva decisión
```

**Importante:** una mayor conversión entre leads contactados y no contactados es evidencia descriptiva.
No prueba por sí sola que el contacto causó la conversión.


## 0. Ejecución en VS Code

Ubicación sugerida:

```text
bd_replica_crm/
└── notebooks/
    ├── 00_command_center_bd_replica_crm.ipynb
    └── 01_decision_intelligence_overview.ipynb
```

Antes de ejecutar:

```powershell
cd C:\Users\user\Documents\dwh\bd_replica_crm
.\.venv\Scripts\Activate.ps1
pip install -e .
```

El notebook usa las conexiones del propio proyecto. No contiene credenciales.


In [1]:
from __future__ import annotations

import sys
import time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError(
        "No encuentro pyproject.toml. Ejecuta el notebook dentro de bd_replica_crm."
    )

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from replica_cygnus.settings import load_settings
from replica_cygnus.connections import connect_postgres

settings = load_settings(PROJECT_ROOT)
conn = connect_postgres(settings)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)

def df(sql: str, params=None) -> pd.DataFrame:
    t0 = time.perf_counter()
    out = pd.read_sql_query(sql, conn, params=params)
    out.attrs["elapsed_s"] = time.perf_counter() - t0
    return out

def exists(schema: str, name: str) -> bool:
    q = df(
        '''
        SELECT EXISTS (
            SELECT 1
            FROM information_schema.tables
            WHERE table_schema=%s AND table_name=%s
            UNION ALL
            SELECT 1
            FROM information_schema.views
            WHERE table_schema=%s AND table_name=%s
        ) AS ok
        ''',
        [schema, name, schema, name],
    )
    return bool(q.iloc[0]["ok"])

print("Repo:", PROJECT_ROOT)
print("Database:", settings.postgres.database)
print("Inicio:", datetime.now().astimezone().isoformat(timespec="seconds"))


Repo: C:\AI\replica_redshift_local\replica_redshift_local
Database: medallio_dw
Inicio: 2026-09-06T22:20:22-05:00


## 1. Contrato de decisión y readiness


In [2]:
objects_expected = [
    ("features", "lead_evidence"),
    ("decision_intelligence", "lead_scores"),
    ("decision_intelligence", "v_lead_priority_current"),
    ("decision_intelligence", "v_lead_score_matured_performance"),
    ("decision_intelligence", "recommendations"),
    ("decision_intelligence", "actions"),
    ("decision_intelligence", "outcomes"),
    ("decision_intelligence", "v_lead_action_outcome"),
    ("decision_intelligence", "v_lead_action_outcome_performance"),
    ("model_control", "model_runs"),
    ("model_control", "model_aliases"),
    ("model_control", "scoring_batches"),
]

readiness = pd.DataFrame(
    [(s, o, exists(s, o)) for s, o in objects_expected],
    columns=["schema", "object", "exists"]
)

readiness


C:\Users\dinat\AppData\Local\Temp\ipykernel_32576\1806747034.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,schema,object,exists
0,features,lead_evidence,True
1,decision_intelligence,lead_scores,True
2,decision_intelligence,v_lead_priority_current,True
3,decision_intelligence,v_lead_score_matured_performance,True
4,decision_intelligence,recommendations,True
5,decision_intelligence,actions,True
6,decision_intelligence,outcomes,True
7,decision_intelligence,v_lead_action_outcome,True
8,decision_intelligence,v_lead_action_outcome_performance,True
9,model_control,model_runs,True


In [3]:
print(
    f"Objetos disponibles: {readiness['exists'].sum()}/{len(readiness)}"
)

missing = readiness.loc[~readiness["exists"], ["schema","object"]]

if len(missing):
    print("\nObjetos aún no disponibles:")
    display(missing)
else:
    print("\n✅ El loop técnico completo está disponible.")


Objetos disponibles: 12/12

✅ El loop técnico completo está disponible.


### Decision contract

Si el contrato existe, mostramos objetivo, unidad de decisión, target, horizonte y responsable.


In [4]:
if exists("decision_intelligence", "decision_contracts"):
    contract = df("""
        SELECT
            decision_system,
            objective,
            decision_unit,
            decision_owner,
            target,
            prediction_horizon_days,
            causal_estimand,
            primary_value_metric,
            feedback_outcome,
            is_active,
            updated_at
        FROM decision_intelligence.decision_contracts
        WHERE decision_system = 'priorizacion_leads'
    """)
    display(contract)
else:
    print("decision_intelligence.decision_contracts no existe todavía.")


C:\Users\dinat\AppData\Local\Temp\ipykernel_32576\1806747034.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,decision_system,objective,decision_unit,decision_owner,target,prediction_horizon_days,causal_estimand,primary_value_metric,feedback_outcome,is_active,updated_at
0,priorizacion_leads,Priorizar atención comercial con evidencia poi...,asignacion de lead (evidence_key),supervisor comercial,minuta_60d,60,"No identificado en v0: propensión, no uplift",tasa de minuta y valor incremental futuro,separacion_14d + minuta_60d,True,2026-09-07 03:10:57.394275+00:00


## 2. Executive Scorecard


In [5]:
def scalar(sql: str, default=np.nan):
    try:
        x = df(sql)
        return x.iloc[0,0] if len(x) else default
    except Exception:
        conn.rollback()
        return default

scorecard = []

if exists("features","lead_evidence"):
    scorecard += [
        ("Evidencias", scalar("SELECT COUNT(*) FROM features.lead_evidence")),
        ("Evidencias LIVE", scalar("SELECT COUNT(*) FROM features.lead_evidence WHERE evidence_source='LIVE'")),
        ("Labels maduros sep.", scalar("SELECT COUNT(*) FROM features.lead_evidence WHERE separacion_14d IS NOT NULL")),
        ("Labels maduros minuta", scalar("SELECT COUNT(*) FROM features.lead_evidence WHERE minuta_60d IS NOT NULL")),
    ]

if exists("decision_intelligence","lead_scores"):
    scorecard += [
        ("Scores", scalar("SELECT COUNT(*) FROM decision_intelligence.lead_scores")),
        ("Último scoring", scalar("SELECT MAX(scored_at) FROM decision_intelligence.lead_scores")),
    ]

if exists("decision_intelligence","recommendations"):
    scorecard.append(
        ("Recomendaciones", scalar(
            "SELECT COUNT(*) FROM decision_intelligence.recommendations "
            "WHERE decision_system='priorizacion_leads'"
        ))
    )

if exists("decision_intelligence","actions"):
    scorecard.append(
        ("Acciones registradas", scalar(
            "SELECT COUNT(*) FROM decision_intelligence.actions"
        ))
    )

if exists("decision_intelligence","outcomes"):
    scorecard.append(
        ("Outcomes registrados", scalar(
            "SELECT COUNT(*) FROM decision_intelligence.outcomes "
            "WHERE decision_system='priorizacion_leads'"
        ))
    )

pd.DataFrame(scorecard, columns=["KPI","Valor"])


C:\Users\dinat\AppData\Local\Temp\ipykernel_32576\1806747034.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,KPI,Valor
0,Evidencias,205947.0
1,Evidencias LIVE,2091.0
2,Labels maduros sep.,204017.0
3,Labels maduros minuta,198719.0
4,Scores,0.0
5,Último scoring,NaN
6,Recomendaciones,0.0
7,Acciones registradas,0.0
8,Outcomes registrados,0.0


## 3. Funnel técnico: evidencia → score → recomendación → acción → outcome


In [6]:
tech_funnel = []

queries = [
    ("Evidencia", "SELECT COUNT(*) FROM features.lead_evidence"),
    ("Score", "SELECT COUNT(DISTINCT evidence_key) FROM decision_intelligence.lead_scores"),
    ("Recomendación",
     "SELECT COUNT(DISTINCT entity_id) FROM decision_intelligence.recommendations "
     "WHERE decision_system='priorizacion_leads'"),
    ("Acción",
     """SELECT COUNT(DISTINCT r.entity_id)
        FROM decision_intelligence.recommendations r
        JOIN decision_intelligence.actions a
          ON a.recommendation_id=r.recommendation_id
        WHERE r.decision_system='priorizacion_leads'"""),
    ("Outcome sep. maduro",
     """SELECT COUNT(*)
        FROM features.lead_evidence
        WHERE separacion_14d IS NOT NULL"""),
    ("Outcome minuta maduro",
     """SELECT COUNT(*)
        FROM features.lead_evidence
        WHERE minuta_60d IS NOT NULL"""),
]

for stage, sql in queries:
    try:
        tech_funnel.append((stage, int(scalar(sql, 0))))
    except Exception:
        conn.rollback()
        tech_funnel.append((stage, np.nan))

tech_funnel = pd.DataFrame(tech_funnel, columns=["stage","n"])
tech_funnel["vs_previous"] = tech_funnel["n"] / tech_funnel["n"].shift(1)
tech_funnel


C:\Users\dinat\AppData\Local\Temp\ipykernel_32576\1806747034.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,stage,n,vs_previous
0,Evidencia,205947,NaN
1,Score,0,0.000000
2,Recomendación,0,NaN
3,Acción,0,NaN
4,Outcome sep. maduro,204017,inf
5,Outcome minuta maduro,198719,0.974032


## 4. Serving model y ejecuciones recientes


In [7]:
if exists("model_control","model_aliases") and exists("model_control","model_runs"):
    aliases = df("""
        SELECT
            a.decision_system,
            a.model_name,
            a.alias_name,
            a.model_run_id,
            mr.model_version,
            mr.status,
            mr.trained_at,
            a.updated_at
        FROM model_control.model_aliases a
        JOIN model_control.model_runs mr USING (model_run_id)
        WHERE a.decision_system='priorizacion_leads'
        ORDER BY a.updated_at DESC
    """)
    display(aliases)
else:
    print("Model registry incompleto.")


C:\Users\dinat\AppData\Local\Temp\ipykernel_32576\1806747034.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,decision_system,model_name,alias_name,model_run_id,model_version,status,trained_at,updated_at


In [8]:
if exists("model_control","scoring_batches"):
    batches = df("""
        SELECT
            scoring_batch_id,
            model_run_id,
            model_version,
            scored_at,
            data_as_of,
            rows_scored,
            status,
            drift_score,
            drift_status,
            notes
        FROM model_control.scoring_batches
        WHERE decision_system='priorizacion_leads'
        ORDER BY scored_at DESC
        LIMIT 30
    """)
    display(batches)
else:
    print("scoring_batches aún no disponible.")


C:\Users\dinat\AppData\Local\Temp\ipykernel_32576\1806747034.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,scoring_batch_id,model_run_id,model_version,scored_at,data_as_of,rows_scored,status,drift_score,drift_status,notes


## 5. Distribución actual de prioridad A/B/C/D


In [9]:
if exists("decision_intelligence","v_lead_priority_current"):
    current = df("""
        SELECT *
        FROM decision_intelligence.v_lead_priority_current
        ORDER BY decision_at DESC, priority_score DESC
    """)

    print("Filas current:", len(current))
    display(current.head(20))
else:
    current = pd.DataFrame()
    print("v_lead_priority_current aún no existe.")


Filas current: 0


C:\Users\dinat\AppData\Local\Temp\ipykernel_32576\1806747034.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,evidence_key,lead_id,documento_cliente,codigo_proyecto,asesor,canal,medio,decision_at,decision_date,scored_at,model_version,model_status,is_provisional,p_separacion_14d,p_minuta_60d,priority_score,priority_rank,priority_band,label_status,separacion_14d,minuta_60d


In [10]:
if len(current):
    bands = (
        current.groupby("priority_band", as_index=False)
        .agg(
            leads=("evidence_key","count"),
            avg_priority_score=("priority_score","mean"),
            avg_p_sep=("p_separacion_14d","mean"),
            avg_p_minuta=("p_minuta_60d","mean"),
        )
    )

    order = pd.Categorical(
        bands["priority_band"],
        categories=["A","B","C","D"],
        ordered=True
    )
    bands = bands.assign(_order=order).sort_values("_order").drop(columns="_order")
    display(bands)


## 6. Calibración: predicho vs observado cuando el outcome ya maduró


In [11]:
if exists("decision_intelligence","v_lead_score_matured_performance"):
    perf = df("""
        SELECT *
        FROM decision_intelligence.v_lead_score_matured_performance
        ORDER BY model_version DESC, priority_band
    """)
    display(perf)
else:
    perf = pd.DataFrame()
    print("v_lead_score_matured_performance aún no existe.")


C:\Users\dinat\AppData\Local\Temp\ipykernel_32576\1806747034.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,model_version,priority_band,n_sep_matured,actual_sep_rate,predicted_sep_rate,n_minuta_matured,actual_minuta_rate,predicted_minuta_rate,avg_priority_score


In [12]:
if len(perf):
    perf2 = perf.copy()
    perf2["sep_calibration_gap_pp"] = (
        perf2["predicted_sep_rate"] - perf2["actual_sep_rate"]
    ) * 100
    perf2["minuta_calibration_gap_pp"] = (
        perf2["predicted_minuta_rate"] - perf2["actual_minuta_rate"]
    ) * 100

    display(perf2[[
        "model_version","priority_band",
        "n_sep_matured","predicted_sep_rate","actual_sep_rate","sep_calibration_gap_pp",
        "n_minuta_matured","predicted_minuta_rate","actual_minuta_rate","minuta_calibration_gap_pp",
        "avg_priority_score"
    ]])


### Interpretación

- **Gap positivo:** el modelo está prediciendo una tasa mayor a la observada.
- **Gap negativo:** la tasa observada está por encima de la predicha.
- No interpretar bandas con pocos outcomes maduros como evidencia estable.


## 7. Backlog de leads accionables


In [13]:
if exists("decision_intelligence","v_lead_action_outcome"):
    backlog = df("""
        SELECT
            recommendation_id,
            evidence_key,
            lead_id,
            decision_date,
            scored_at,
            priority_band,
            priority_score,
            p_minuta_60d,
            recommended_action,
            action_status,
            action_taken,
            action_owner,
            action_at
        FROM decision_intelligence.v_lead_action_outcome
        WHERE action_status='NOT_RECORDED'
        ORDER BY
            CASE priority_band
                WHEN 'A' THEN 1
                WHEN 'B' THEN 2
                WHEN 'C' THEN 3
                WHEN 'D' THEN 4
                ELSE 9
            END,
            priority_score DESC,
            scored_at DESC
        LIMIT 100
    """)
    print("Top backlog accionable:")
    display(backlog.head(30))
else:
    backlog = pd.DataFrame()
    print("v_lead_action_outcome aún no disponible.")


Top backlog accionable:


C:\Users\dinat\AppData\Local\Temp\ipykernel_32576\1806747034.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,recommendation_id,evidence_key,lead_id,decision_date,scored_at,priority_band,priority_score,p_minuta_60d,recommended_action,action_status,action_taken,action_owner,action_at


In [14]:
if len(backlog):
    backlog_summary = (
        backlog.groupby(
            ["priority_band","recommended_action"],
            dropna=False,
            as_index=False
        )
        .agg(
            backlog=("recommendation_id","count"),
            avg_score=("priority_score","mean"),
            avg_p_minuta=("p_minuta_60d","mean"),
        )
    )
    display(backlog_summary)


## 8. Cobertura operacional de acción


In [15]:
if exists("decision_intelligence","v_lead_action_outcome"):
    action_coverage = df("""
        SELECT
            priority_band,
            COUNT(*) AS recommendations,
            COUNT(action_id) AS actions_recorded,
            AVG(CASE WHEN action_id IS NOT NULL THEN 1.0 ELSE 0.0 END) AS action_coverage,
            AVG(CASE
                WHEN followed_recommendation IS TRUE THEN 1.0
                WHEN followed_recommendation IS FALSE THEN 0.0
                ELSE NULL
            END) AS recommendation_follow_rate,
            AVG(EXTRACT(EPOCH FROM (action_at - scored_at))/3600.0)
                FILTER (WHERE action_at IS NOT NULL) AS avg_hours_to_action
        FROM decision_intelligence.v_lead_action_outcome
        GROUP BY priority_band
        ORDER BY CASE priority_band
            WHEN 'A' THEN 1
            WHEN 'B' THEN 2
            WHEN 'C' THEN 3
            WHEN 'D' THEN 4
            ELSE 9
        END
    """)
    display(action_coverage)
else:
    action_coverage = pd.DataFrame()


C:\Users\dinat\AppData\Local\Temp\ipykernel_32576\1806747034.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,priority_band,recommendations,actions_recorded,action_coverage,recommendation_follow_rate,avg_hours_to_action


### El KPI más importante aquí

Antes de preguntar “¿el modelo convierte más?”, hay que preguntar:

> **¿Qué porcentaje de las recomendaciones realmente llega a una acción humana registrada?**

Si `action_coverage` es bajo, todavía estamos midiendo principalmente **predicción**, no intervención.


## 9. Outcome observacional por banda y acción


In [16]:
if exists("decision_intelligence","v_lead_action_outcome_performance"):
    action_perf = df("""
        SELECT *
        FROM decision_intelligence.v_lead_action_outcome_performance
        ORDER BY decision_date DESC, priority_band, action_status, action_taken
    """)
    display(action_perf.head(50))
else:
    action_perf = pd.DataFrame()


C:\Users\dinat\AppData\Local\Temp\ipykernel_32576\1806747034.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,decision_date,priority_band,action_status,action_taken,recommendations,actions_recorded,sep_matured,sep_rate,minuta_matured,minuta_rate,total_action_cost


In [17]:
if len(action_perf):
    matured_action_summary = (
        action_perf.groupby(
            ["priority_band","action_status","action_taken"],
            dropna=False,
            as_index=False
        )
        .agg(
            recommendations=("recommendations","sum"),
            actions_recorded=("actions_recorded","sum"),
            sep_matured=("sep_matured","sum"),
            minuta_matured=("minuta_matured","sum"),
            total_action_cost=("total_action_cost","sum"),
        )
    )

    # Recalcular tasas correctamente requiere numeradores.
    # Usamos la vista detalle para evitar promediar tasas agregadas.
    detail = df("""
        SELECT *
        FROM decision_intelligence.v_lead_action_outcome
    """)

    detail_summary = (
        detail.groupby(
            ["priority_band","action_status","action_taken"],
            dropna=False,
            as_index=False
        )
        .agg(
            recommendations=("recommendation_id","count"),
            actions_recorded=("action_id", lambda s: s.notna().sum()),
            sep_matured=("separacion_14d", lambda s: s.notna().sum()),
            sep_rate=("separacion_14d","mean"),
            minuta_matured=("minuta_60d", lambda s: s.notna().sum()),
            minuta_rate=("minuta_60d","mean"),
            avg_action_cost=("action_cost","mean"),
        )
        .sort_values(
            ["priority_band","recommendations"],
            ascending=[True,False]
        )
    )

    display(detail_summary)


### Advertencia causal

La tabla anterior puede responder:

- qué ocurrió después de una recomendación;
- qué ocurrió cuando hubo/no hubo acción registrada;
- qué bandas convierten más;
- cuánto costaron las acciones.

**No puede responder todavía** cuánto de esa diferencia fue causado por la acción.
Para eso necesitaremos asignación experimental o un diseño causal válido.


## 10. Segmentación por proyecto, asesor, canal y medio


In [18]:
if exists("features","lead_evidence"):
    evidence = df("""
        SELECT
            evidence_key,
            lead_id,
            decision_at,
            evidence_source,
            codigo_proyecto,
            asesor,
            canal,
            medio,
            project_sep_rate_90d,
            project_minuta_rate_180d,
            advisor_sep_rate_90d,
            advisor_minuta_rate_180d,
            global_sep_rate_90d,
            global_minuta_rate_180d,
            separacion_14d,
            minuta_60d,
            label_status
        FROM features.lead_evidence
    """)
    print("Evidencias:", len(evidence))
else:
    evidence = pd.DataFrame()


C:\Users\dinat\AppData\Local\Temp\ipykernel_32576\1806747034.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


Evidencias: 205947


In [19]:
def segment_summary(data: pd.DataFrame, col: str, min_n: int = 10):
    if data.empty or col not in data.columns:
        return pd.DataFrame()

    x = (
        data.groupby(col, dropna=False, as_index=False)
        .agg(
            leads=("evidence_key","count"),
            sep_matured=("separacion_14d", lambda s: s.notna().sum()),
            sep_rate=("separacion_14d","mean"),
            minuta_matured=("minuta_60d", lambda s: s.notna().sum()),
            minuta_rate=("minuta_60d","mean"),
        )
    )

    return (
        x[x["leads"] >= min_n]
        .sort_values(["leads","minuta_rate"], ascending=False)
    )

for dimension in ["codigo_proyecto","asesor","canal","medio"]:
    print("\n###", dimension)
    display(segment_summary(evidence, dimension).head(25))



### codigo_proyecto


,codigo_proyecto,leads,sep_matured,sep_rate,minuta_matured,minuta_rate
9,MA,21248,21245,0.000800,21234,0.000141
4,CUBA,19529,19529,0.000819,19523,0.000154
3,CRUZ,18439,18439,0.000488,18432,0.000054
8,GY,16508,16363,0.002017,15911,0.001823
17,URT,13556,13555,0.000221,13550,0.000148
12,NP,13337,12885,0.008304,11597,0.007588
6,ES,12380,12380,0.000323,12377,0.000162
5,EEUU,11950,11874,0.001937,11652,0.001287
7,FX,11573,11485,0.003918,11123,0.002877
2,CP,11558,10995,0.002819,9961,0.002510



### asesor


,asesor,leads,sep_matured,sep_rate,minuta_matured,minuta_rate
0,NaN,205947,204017,0.002358,198719,0.001766



### canal


,canal,leads,sep_matured,sep_rate,minuta_matured,minuta_rate
0,NaN,205947,204017,0.002358,198719,0.001766



### medio


,medio,leads,sep_matured,sep_rate,minuta_matured,minuta_rate
9,facebook,133755,132386,0.000151,128623,0.000109
21,verdancy,9603,9603,0.000208,9595,0.000104
12,instagram,9504,9488,0.002846,9451,0.002434
4,contacto web,8626,8623,0.002087,8612,0.001510
16,paso por la zona,8346,8200,0.009756,7778,0.007843
3,caseta de ventas,7793,7754,0.017152,7537,0.011145
15,nexo,7145,7100,0.005915,6921,0.004768
23,NaN,6224,6156,0.002112,5932,0.002023
19,tik tok,4276,4264,0.000469,4258,0.000235
10,feria inmobiliaria,3052,2908,0.027854,2692,0.022288


## 11. Opportunity Map: volumen × conversión


In [20]:
if len(evidence):
    project_opportunity = segment_summary(
        evidence, "codigo_proyecto", min_n=20
    )

    if len(project_opportunity):
        global_minuta = evidence["minuta_60d"].mean()
        project_opportunity["global_minuta_rate"] = global_minuta
        project_opportunity["gap_vs_global_pp"] = (
            project_opportunity["minuta_rate"] - global_minuta
        ) * 100

        project_opportunity["opportunity_score"] = (
            project_opportunity["leads"]
            * (global_minuta - project_opportunity["minuta_rate"]).clip(lower=0)
        )

        display(
            project_opportunity.sort_values(
                "opportunity_score", ascending=False
            ).head(25)
        )


,codigo_proyecto,leads,sep_matured,sep_rate,minuta_matured,minuta_rate,global_minuta_rate,gap_vs_global_pp,opportunity_score
9,MA,21248,21245,0.000800,21234,0.000141,0.001766,-0.162503,34.528646
3,CRUZ,18439,18439,0.000488,18432,0.000054,0.001766,-0.171206,31.568670
4,CUBA,19529,19529,0.000819,19523,0.000154,0.001766,-0.161265,31.493409
17,URT,13556,13555,0.000221,13550,0.000148,0.001766,-0.161871,21.943257
6,ES,12380,12380,0.000323,12377,0.000162,0.001766,-0.160472,19.866473
14,TP00,10128,10125,0.000000,10116,0.000000,0.001766,-0.176631,17.889220
16,UN,4528,4528,0.000221,4527,0.000000,0.001766,-0.176631,7.997866
0,001,4372,4372,0.001144,4366,0.000229,0.001766,-0.153727,6.720947
5,EEUU,11950,11874,0.001937,11652,0.001287,0.001766,-0.047898,5.723818
1,CAM,676,676,0.000000,618,0.000000,0.001766,-0.176631,1.194028


`opportunity_score` es deliberadamente simple:

```text
volumen de leads × brecha negativa vs conversión global
```

No es un modelo causal ni una recomendación automática.
Sirve para priorizar dónde investigar primero.


## 12. Cohortes temporales


In [21]:
if len(evidence):
    cohort = evidence.copy()
    cohort["decision_at"] = pd.to_datetime(
        cohort["decision_at"], errors="coerce", utc=True
    )
    cohort["week"] = cohort["decision_at"].dt.to_period("W").astype(str)

    cohort_week = (
        cohort.groupby("week", as_index=False)
        .agg(
            leads=("evidence_key","count"),
            sep_matured=("separacion_14d", lambda s: s.notna().sum()),
            sep_rate=("separacion_14d","mean"),
            minuta_matured=("minuta_60d", lambda s: s.notna().sum()),
            minuta_rate=("minuta_60d","mean"),
        )
        .sort_values("week")
    )

    display(cohort_week.tail(30))


C:\Users\dinat\AppData\Local\Temp\ipykernel_32576\2841238535.py:6: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  cohort["week"] = cohort["decision_at"].dt.to_period("W").astype(str)


,week,leads,sep_matured,sep_rate,minuta_matured,minuta_rate
333,2026-02-16/2026-02-22,1529,1529,0.003924,1529,0.002616
334,2026-02-23/2026-03-01,1049,1049,0.013346,1049,0.013346
335,2026-03-02/2026-03-08,905,905,0.003315,905,0.003315
336,2026-03-09/2026-03-15,1417,1417,0.016937,1417,0.014820
337,2026-03-16/2026-03-22,917,917,0.014177,917,0.009815
338,2026-03-23/2026-03-29,1818,1818,0.002750,1818,0.002200
339,2026-03-30/2026-04-05,1216,1216,0.004934,1216,0.004934
340,2026-04-06/2026-04-12,1269,1269,0.004728,1269,0.004728
341,2026-04-13/2026-04-19,1172,1172,0.003413,1172,0.005119
342,2026-04-20/2026-04-26,1161,1161,0.010336,1161,0.011197


## 13. Predicción vs base histórica por proyecto


In [22]:
if len(current):
    project_prediction = (
        current.groupby("codigo_proyecto", dropna=False, as_index=False)
        .agg(
            scored_leads=("evidence_key","count"),
            avg_priority_score=("priority_score","mean"),
            avg_pred_sep=("p_separacion_14d","mean"),
            avg_pred_minuta=("p_minuta_60d","mean"),
            historical_sep_rate=("separacion_14d","mean"),
            historical_minuta_rate=("minuta_60d","mean"),
        )
        .sort_values("scored_leads", ascending=False)
    )

    display(project_prediction.head(30))


## 14. Calidad del scoring operativo


In [23]:
if len(current):
    scoring_quality = pd.DataFrame({
        "check": [
            "priority_score fuera de [0,100]",
            "p_separacion_14d fuera de [0,1]",
            "p_minuta_60d fuera de [0,1]",
            "priority_band inválida",
            "evidence_key duplicado en current",
            "lead_id nulo",
            "codigo_proyecto nulo",
        ],
        "n": [
            ((current["priority_score"] < 0) | (current["priority_score"] > 100)).sum(),
            ((current["p_separacion_14d"] < 0) | (current["p_separacion_14d"] > 1)).sum(),
            ((current["p_minuta_60d"] < 0) | (current["p_minuta_60d"] > 1)).sum(),
            (~current["priority_band"].isin(["A","B","C","D"])).sum(),
            current["evidence_key"].duplicated().sum(),
            current["lead_id"].isna().sum(),
            current["codigo_proyecto"].isna().sum(),
        ]
    })

    scoring_quality["status"] = np.where(
        scoring_quality["n"].eq(0), "OK", "REVISAR"
    )
    scoring_quality


## 15. Concentración: ¿el top scoring realmente concentra outcomes?


In [24]:
if len(current):
    matured = current[
        current["minuta_60d"].notna()
    ].copy()

    if len(matured) >= 20:
        matured = matured.sort_values(
            "priority_score", ascending=False
        ).reset_index(drop=True)

        matured["pct_population"] = (
            np.arange(1, len(matured)+1) / len(matured)
        )

        total_minutas = matured["minuta_60d"].sum()

        if total_minutas > 0:
            matured["cum_minutas"] = matured["minuta_60d"].cumsum()
            matured["pct_minutas_captured"] = (
                matured["cum_minutas"] / total_minutas
            )

            lift_points = []
            for frac in [0.10,0.20,0.30,0.50]:
                subset = matured.head(max(1, int(len(matured)*frac)))
                captured = subset["minuta_60d"].sum() / total_minutas
                lift_points.append({
                    "top_fraction": frac,
                    "population_n": len(subset),
                    "minutas_captured_pct": captured,
                    "capture_vs_random": captured / frac,
                })

            lift_table = pd.DataFrame(lift_points)
            display(lift_table)
        else:
            print("No hay minutas positivas maduras en current.")
    else:
        print("Todavía hay pocos outcomes maduros para concentración estable.")


### Esta tabla es especialmente útil

Si, por ejemplo, el **top 20%** por score concentra claramente más de 20% de las minutas maduras,
el ranking está ordenando valor predictivo.

Eso todavía no significa que actuar sobre ese top 20% genere uplift causal.
Sí demuestra que el score puede ser operacionalmente útil para **priorizar atención escasa**.


## 16. Smart Insights automáticos


In [25]:
insights = []

if len(current):
    band_counts = current["priority_band"].value_counts()
    total = len(current)
    pct_a = band_counts.get("A",0) / total if total else 0
    insights.append(
        f"{pct_a:.1%} del universo current está clasificado como prioridad A."
    )

if len(action_coverage):
    total_rec = action_coverage["recommendations"].sum()
    total_actions = action_coverage["actions_recorded"].sum()
    coverage = total_actions / total_rec if total_rec else np.nan

    if pd.notna(coverage):
        insights.append(
            f"La cobertura operativa de acción registrada es {coverage:.1%}."
        )

        if coverage < 0.50:
            insights.append(
                "La principal restricción actual parece ser de ejecución/registro: "
                "menos de la mitad de las recomendaciones tiene acción registrada."
            )

if len(perf):
    p = perf.copy()
    p = p[p["n_minuta_matured"].fillna(0) >= 10]
    if len(p):
        best = p.sort_values(
            "actual_minuta_rate", ascending=False
        ).iloc[0]
        insights.append(
            f"La banda con mayor minuta observada entre grupos con >=10 maduros es "
            f"{best['priority_band']} ({best['actual_minuta_rate']:.1%})."
        )

if len(backlog):
    high = backlog["priority_band"].isin(["A","B"]).sum()
    insights.append(
        f"Hay {high:,} recomendaciones A/B sin acción registrada dentro del backlog mostrado."
    )

if len(evidence):
    mature_minuta = evidence["minuta_60d"].notna().mean()
    insights.append(
        f"{mature_minuta:.1%} de las evidencias ya tiene horizonte de minuta observable."
    )

print("=== SMART INSIGHTS ===")
for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")


=== SMART INSIGHTS ===
1. 96.5% de las evidencias ya tiene horizonte de minuta observable.


## 17. Gate de madurez: ¿qué podemos afirmar hoy?


In [26]:
gates = []

gates.append({
    "gate": "Datos point-in-time",
    "status": "PASS" if exists("features","lead_evidence") else "PENDING",
    "evidence": "features.lead_evidence"
})

gates.append({
    "gate": "Predicción persistida",
    "status": "PASS" if exists("decision_intelligence","lead_scores") else "PENDING",
    "evidence": "decision_intelligence.lead_scores"
})

gates.append({
    "gate": "Recomendación persistida",
    "status": "PASS" if exists("decision_intelligence","recommendations") else "PENDING",
    "evidence": "decision_intelligence.recommendations"
})

gates.append({
    "gate": "Acción humana trazable",
    "status": "PASS" if exists("decision_intelligence","actions") else "PENDING",
    "evidence": "decision_intelligence.actions"
})

gates.append({
    "gate": "Outcome trazable",
    "status": "PASS" if exists("decision_intelligence","outcomes") else "PENDING",
    "evidence": "decision_intelligence.outcomes"
})

has_mature_outcomes = False
if len(evidence):
    has_mature_outcomes = bool(
        evidence["separacion_14d"].notna().any()
        or evidence["minuta_60d"].notna().any()
    )

gates.append({
    "gate": "Outcomes maduros reales",
    "status": "PASS" if has_mature_outcomes else "PENDING",
    "evidence": "separacion_14d / minuta_60d"
})

gates.append({
    "gate": "Efecto causal de la acción",
    "status": "NOT_YET",
    "evidence": "Requiere experimento o diseño causal aprobado"
})

gate_table = pd.DataFrame(gates)
gate_table


C:\Users\dinat\AppData\Local\Temp\ipykernel_32576\1806747034.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,gate,status,evidence
0,Datos point-in-time,PASS,features.lead_evidence
1,Predicción persistida,PASS,decision_intelligence.lead_scores
2,Recomendación persistida,PASS,decision_intelligence.recommendations
3,Acción humana trazable,PASS,decision_intelligence.actions
4,Outcome trazable,PASS,decision_intelligence.outcomes
5,Outcomes maduros reales,PASS,separacion_14d / minuta_60d
6,Efecto causal de la acción,NOT_YET,Requiere experimento o diseño causal aprobado


## 18. Qué debería hacer después

La secuencia recomendada es:

```text
1. Verificar evidencia real
2. Verificar scoring
3. Verificar distribución A/B/C/D
4. Verificar que las recomendaciones existan
5. Aumentar cobertura de acciones registradas
6. Esperar/madurar outcomes
7. Medir performance predictivo
8. Diseñar experimento / uplift
9. Convertir resultado en política de decisión versionada
```

El cuello de botella deja de ser “tener un modelo”.
Pasa a ser **cerrar consistentemente el loop de decisión**.


## 19. Export para Power BI / comité


In [27]:
EXPORT = False

if EXPORT:
    out = PROJECT_ROOT / "reports" / "decision_intelligence_overview"
    out.mkdir(parents=True, exist_ok=True)

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    datasets = {
        "readiness": readiness,
        "tech_funnel": tech_funnel,
        "current_priority": current,
        "matured_performance": perf,
        "action_coverage": action_coverage,
        "action_performance": action_perf,
        "backlog": backlog,
        "evidence": evidence,
        "gates": gate_table,
    }

    for name, data in datasets.items():
        if isinstance(data, pd.DataFrame) and len(data):
            data.to_csv(
                out / f"{name}_{stamp}.csv",
                index=False
            )

    print("Exportado en:", out)
else:
    print("EXPORT=False. Cambia a True para exportar snapshots.")


EXPORT=False. Cambia a True para exportar snapshots.


## 20. Cierre


In [28]:
conn.close()
print("Conexión PostgreSQL cerrada.")


Conexión PostgreSQL cerrada.
